In [12]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [13]:
df = pd.read_csv("../data/diabetes.csv")

print("Columns:", df.columns.tolist())
print(df.head())


Columns: ['gender', 'age', 'hypertension', 'heart_disease', 'smoking_history', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'diabetes']
   gender   age  hypertension  heart_disease smoking_history    bmi  \
0  Female  80.0             0              1           never  25.19   
1  Female  54.0             0              0         No Info  27.32   
2    Male  28.0             0              0           never  27.32   
3  Female  36.0             0              0         current  23.45   
4    Male  76.0             1              1         current  20.14   

   HbA1c_level  blood_glucose_level  diabetes  
0          6.6                  140         0  
1          6.6                   80         0  
2          5.7                  158         0  
3          5.0                  155         0  
4          4.8                  155         0  


In [14]:
df = df.drop(["gender", "smoking_history"], axis=1)

# convert all column names to lowercase
df.columns = df.columns.str.lower()


In [15]:
X = df.drop("diabetes", axis=1)
y = df["diabetes"]


In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [17]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [18]:
selector = SelectKBest(score_func=f_classif, k=5)

X_train = selector.fit_transform(X_train, y_train)
X_test = selector.transform(X_test)


In [19]:
lr_model = LogisticRegression()
lr_model.fit(X_train, y_train)

rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)


def evaluate_model(model, X, y):
    y_pred = model.predict(X)

    return {
        "Accuracy": accuracy_score(y, y_pred),
        "Precision": precision_score(y, y_pred),
        "Recall": recall_score(y, y_pred),
        "F1": f1_score(y, y_pred)
    }


lr_results = evaluate_model(lr_model, X_test, y_test)
rf_results = evaluate_model(rf_model, X_test, y_test)

print("\nLogistic Regression:", lr_results)
print("Random Forest:", rf_results)


best_model = rf_model if rf_results["F1"] > lr_results["F1"] else lr_model

print("\nBest Model Selected:",
      "Random Forest" if best_model == rf_model else "Logistic Regression")



Logistic Regression: {'Accuracy': 0.9588, 'Precision': 0.867109634551495, 'Recall': 0.6112412177985949, 'F1': 0.717032967032967}
Random Forest: {'Accuracy': 0.96785, 'Precision': 0.9043280182232346, 'Recall': 0.6973067915690867, 'F1': 0.7874380165289256}

Best Model Selected: Random Forest


In [20]:
final_model = best_model

y_probs = final_model.predict_proba(X_test)[:, 1]

best_threshold = 0.5
best_f1 = 0

for thresh in np.arange(0.3, 0.91, 0.05):
    y_pred_thresh = (y_probs >= thresh).astype(int)
    f1 = f1_score(y_test, y_pred_thresh)

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = thresh

print("\nBest Threshold:", best_threshold)
print("Best F1 Score:", best_f1)

y_final_pred = (y_probs >= best_threshold).astype(int)

print("\nFinal Evaluation:")
print("Accuracy:", accuracy_score(y_test, y_final_pred))
print("Precision:", precision_score(y_test, y_final_pred))
print("Recall:", recall_score(y_test, y_final_pred))
print("F1 Score:", f1_score(y_test, y_final_pred))



Best Threshold: 0.8999999999999999
Best F1 Score: 0.8053128276826285

Final Evaluation:
Accuracy: 0.97215
Precision: 0.9991326973113617
Recall: 0.6744730679156908
F1 Score: 0.8053128276826285


In [21]:
model_data = {
    "model": final_model,
    "scaler": scaler,
    "selector": selector,
    "threshold": best_threshold
}

joblib.dump(model_data, "../models/diabetes_model.pkl")

print("\nModel saved successfully!")



Model saved successfully!
